# Fraud Analysis with Blind Insight

This notebook demonstrates batch aggregation analysis of encrypted fraud detection data using the Blind Insight API.

## Overview
- **Purpose**: Analyze fraud detection data using encrypted aggregations
- **Data**: Fraud analysis training dataset
- **Method**: Batch-wise aggregation without decrypting individual records

In [51]:
# Standard library imports
import sys
import warnings

# Third-party imports
import pandas as pd
from sklearn.linear_model import LogisticRegression

# Blind Insight client import
sys.path.append('.')
from blind_insight_client import BlindInsightClient

# Suppress SSL warnings for local development
warnings.filterwarnings('ignore', message='Unverified HTTPS request')


In [52]:
# ============================================================================
# CONFIGURATION
# ============================================================================
# Blind Insight API configuration
ORGANIZATION = "demo"
API_URL = "https://proxy.local.blindinsight.io/"
USERNAME = "data_owner@localhost"
PASSWORD = "blindinsight"

# Dataset and schema identifiers
SCHEMA_SLUG = "fraud-analysis-schema"
TRAINING_DATASET_SLUG = "fraud-analysis-training"
REAL_DATASET_SLUG = "fraud-analysis"
TRAINING_SCHEMA_ID = "iEbXomadeyp3JcDfZMFLYp"
REAL_DATA_SCHEMA_ID = "i9yp3TutxNsCp3fgNWE4p2"

In [53]:
# ============================================================================
# HELPER FUNCTIONS
# ============================================================================

def agg_value(resp):
    recs = resp.get("records", [])
    if not recs:
        raise ValueError(f"Unexpected aggregation response: {resp}")
    rec0 = recs[0]
    if "data" in rec0 and isinstance(rec0["data"], dict) and "value" in rec0["data"]:
        v = rec0["data"].get("value")
        return float(v) if v is not None else 0.0
    if "value" in rec0:
        v = rec0.get("value")
        return float(v) if v is not None else 0.0
    raise ValueError(f"Unexpected aggregation response shape: {resp}")


def get_max_range(feature, schema):
    return schema["properties"].get(feature, {}).get("maximum", 1)

# Batch Aggregation Analysis

This section demonstrates batch-wise aggregation of encrypted data features using the Blind Insight API.

In [54]:
# Initialize client
client = BlindInsightClient(api_url=API_URL, username=USERNAME, password=PASSWORD, verify_ssl=False)

# Define features and schema for fraud analysis
features = [
    "amount",
    "transaction_type_ATM",
    "transaction_type_Online",
    "transaction_type_POS",
    "transaction_type_QR",
    "merchant_category_Clothing",
    "merchant_category_Electronics",
    "merchant_category_Food",
    "merchant_category_Grocery",
    "merchant_category_Travel",
    "country_DE",
    "country_FR",
    "country_NG",
    "country_TR",
    "country_UK",
    "country_US",
    "hour",
    "is_fraud",
]

schema = {
    "type": "object",
    "properties": {
        "hour": {"type": "integer", "maximum": 23, "minimum": 0},
        "amount": {"type": "integer", "maximum": 12000, "minimum": 0},
        "user_id": {"type": "integer", "maximum": 1000, "minimum": 0},
        "is_fraud": {"type": "integer", "maximum": 1, "minimum": 0},
        "country_DE": {"type": "integer", "maximum": 1, "minimum": 0},
        "country_FR": {"type": "integer", "maximum": 1, "minimum": 0},
        "country_NG": {"type": "integer", "maximum": 1, "minimum": 0},
        "country_TR": {"type": "integer", "maximum": 1, "minimum": 0},
        "country_UK": {"type": "integer", "maximum": 1, "minimum": 0},
        "country_US": {"type": "integer", "maximum": 1, "minimum": 0},
        "dataset_order": {"type": "integer", "maximum": 500, "minimum": 0},
        "ip_risk_score": {"type": "integer", "maximum": 100, "minimum": 0},
        "transaction_id": {"type": "integer", "maximum": 10000, "minimum": 0},
        "device_risk_score": {"type": "integer", "maximum": 100, "minimum": 0},
        "transaction_type_QR": {"type": "integer", "maximum": 1, "minimum": 0},
        "transaction_type_ATM": {"type": "integer", "maximum": 1, "minimum": 0},
        "transaction_type_POS": {"type": "integer", "maximum": 1, "minimum": 0},
        "merchant_category_Food": {"type": "integer", "maximum": 1, "minimum": 0},
        "transaction_type_Online": {"type": "integer", "maximum": 1, "minimum": 0},
        "merchant_category_Travel": {"type": "integer", "maximum": 1, "minimum": 0},
        "merchant_category_Grocery": {"type": "integer", "maximum": 1, "minimum": 0},
        "merchant_category_Clothing": {"type": "integer", "maximum": 1, "minimum": 0},
        "merchant_category_Electronics": {"type": "integer", "maximum": 1, "minimum": 0}
    }
}

# Configuration
batch_size = 10
SCHEMA_ID = TRAINING_SCHEMA_ID
DATASET_SLUG = TRAINING_DATASET_SLUG

# Get total record count
print("Getting total record count...")
count_resp = client.aggregate(
    organization=ORGANIZATION,
    dataset_slug=DATASET_SLUG,
    schema_slug=SCHEMA_SLUG,
    agg_filter="dataset_order:count(0~100000)",
    decrypt=False,
    schema_id=SCHEMA_ID
)
total_count = int(agg_value(count_resp))
print(f"Total records: {total_count}")
total_rows = total_count

# Compute overall means for first 5 features (sample)
print(f"\nComputing mean for each feature across all {total_count} records:\n")
for feature in features[:5]:
    try:
        max_range = get_max_range(feature, schema)
        resp = client.aggregate(
            organization=ORGANIZATION,
            dataset_slug=DATASET_SLUG,
            schema_slug=SCHEMA_SLUG,
            agg_filter=f"{feature}:avg(0~{max_range})",
            decrypt=False,
            schema_id=SCHEMA_ID
        )
        mean_val = agg_value(resp)
        print(f"  mean {feature}: {mean_val:.3f}")
    except Exception as e:
        print(f"  Error with {feature}: {str(e)[:100]}")

# Batch aggregation
print(f"\nStarting batch mean aggregation for each feature using batch size {batch_size}:\n")
batch_data = []

start = 0
while start < total_rows:
    end = min(start + batch_size - 1, total_rows - 1)
    batch_filter = f"dataset_order:{start}~{end}"
    print(f"\nBatch {start}-{end}:")

    data = {}
    for feature in features:
        max_range = get_max_range(feature, schema)
        resp = client.aggregate(
            organization=ORGANIZATION,
            dataset_slug=DATASET_SLUG,
            schema_slug=SCHEMA_SLUG,
            agg_filter=f"{feature}:avg(0~{max_range})",
            extra_filters=[batch_filter],
            decrypt=False,
            schema_id=SCHEMA_ID
        )
        mean_val = agg_value(resp)
        data[feature] = mean_val
        print(f"  mean {feature}: {mean_val:.3f}")
    start += batch_size
    batch_data.append(data)

# Create DataFrame from batch results
df_batch_means = pd.DataFrame(batch_data)
print(f"\nBatch aggregation results:\n{df_batch_means}")
print("\nAggregation complete!")

Getting total record count...
Total records: 500

Computing mean for each feature across all 500 records:

  mean amount: 916.804
  mean transaction_type_ATM: 0.250
  mean transaction_type_Online: 0.266
  mean transaction_type_POS: 0.234
  mean transaction_type_QR: 0.250

Starting batch mean aggregation for each feature using batch size 10:


Batch 0-9:
  mean amount: 2511.100
  mean transaction_type_ATM: 0.400
  mean transaction_type_Online: 0.100
  mean transaction_type_POS: 0.200
  mean transaction_type_QR: 0.300
  mean merchant_category_Clothing: 0.100
  mean merchant_category_Electronics: 0.100
  mean merchant_category_Food: 0.300
  mean merchant_category_Grocery: 0.100
  mean merchant_category_Travel: 0.400
  mean country_DE: 0.200
  mean country_FR: 0.100
  mean country_NG: 0.200
  mean country_TR: 0.300
  mean country_UK: 0.100
  mean country_US: 0.100
  mean hour: 13.400
  mean is_fraud: 1.000

Batch 10-19:
  mean amount: 1251.200
  mean transaction_type_ATM: 0.600
  mean tran

## Privacy-Preserving Machine Learning

The DataFrame `df_batch_means` contains **aggregated batch statistics** computed from encrypted data without decrypting individual records. This demonstrates **privacy-preserving machine learning** where:

- Each row represents aggregated statistics for a batch of transactions
- The target variable `is_fraud` is the mean fraud rate per batch (0.0-1.0)
- Models can be trained on these aggregated statistics without accessing individual transaction data
- This approach preserves privacy while enabling machine learning insights

This is the core value proposition of Blind Insight: **train models using encrypted aggregations without ever decrypting sensitive individual records**.

In [55]:
df_features = df_batch_means.iloc[:, :-1]  # All columns except last
df_target = df_batch_means.iloc[:, -1]     # Last column (is_fraud mean per batch)

# Check class distribution
print(f"Target value range: {df_target.min():.3f} to {df_target.max():.3f}")
print(f"Unique target values: {sorted(df_target.unique())}")

# Train model on aggregated statistics (privacy-preserving ML)
model = LogisticRegression(max_iter=1000, random_state=42)
model.fit(df_features, df_target)
print("Model trained successfully on aggregated batch statistics (encrypted data)")

Target value range: 0.000 to 1.000
Unique target values: [np.float64(0.0), np.float64(1.0)]
Model trained successfully on aggregated batch statistics (encrypted data)


In [56]:
# Evaluate model on aggregated batch data
score = model.score(df_features, df_target)
print(f"Model accuracy on batch aggregation data: {score:.4f}")

Model accuracy on batch aggregation data: 1.0000


In [57]:
# Test with modified target to create class diversity
# This is for testing purposes only - demonstrates the need for class diversity
df_temp = df_target.copy()
df_temp.iloc[0] = 0.0  # Modify first value to create class diversity
print("Modified target values (first value set to 0.0):")
print(df_temp.head(10))

Modified target values (first value set to 0.0):
0    0.0
1    1.0
2    1.0
3    1.0
4    1.0
5    1.0
6    1.0
7    1.0
8    1.0
9    1.0
Name: is_fraud, dtype: float64


In [58]:
# Retrain with modified target
model_test = LogisticRegression(max_iter=1000, random_state=42)
model_test.fit(df_features, df_temp)
score = model_test.score(df_features, df_temp)
print(f"Model accuracy with modified target: {score:.4f}")

Model accuracy with modified target: 0.9800
